In [1]:
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")
print(repr(API_KEY))

'43fe5f9e72e42cb69d11fce16250c956'


In [6]:
def extract(city: str) -> dict:
    url = f"https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "imperial"  # or "metric"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()  # raises error on bad status
    return response.json()

# Test it
raw = extract("Tampa,FL,US")
print(raw)

{'coord': {'lon': -82.4584, 'lat': 27.9475}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 82.4, 'feels_like': 84.81, 'temp_min': 80.56, 'temp_max': 84.96, 'pressure': 1016, 'humidity': 59, 'sea_level': 1016, 'grnd_level': 1014}, 'visibility': 10000, 'wind': {'speed': 14.97, 'deg': 50}, 'clouds': {'all': 20}, 'dt': 1777331331, 'sys': {'type': 2, 'id': 2077775, 'country': 'US', 'sunrise': 1777287196, 'sunset': 1777334483}, 'timezone': -14400, 'id': 4174757, 'name': 'Tampa', 'cod': 200}


In [3]:
def transform(raw: dict) -> pd.DataFrame:
    record = {
        "city":        raw["name"],
        "temp_f":      raw["main"]["temp"],
        "feels_like_f": raw["main"]["feels_like"],
        "humidity":    raw["main"]["humidity"],
        "condition":   raw["weather"][0]["description"],
        "wind_mph":    raw["wind"]["speed"],
        "fetched_at":  datetime.utcnow().isoformat()
    }
    df = pd.DataFrame([record])

    # Basic validation
    assert df["temp_f"].between(-80, 140).all(), "Temp out of range!"
    return df

clean_df = transform(raw)
print(clean_df)

    city  temp_f  feels_like_f  humidity         condition  wind_mph  \
0  Tampa   84.76          87.4        55  scattered clouds      13.8   

                   fetched_at  
0  2026-04-27T22:45:28.918891  


In [4]:
def load(df: pd.DataFrame, db_path: str = "weather.db"):
    engine = create_engine(f"sqlite:///{db_path}")
    df.to_sql(
        name="weather_readings",
        con=engine,
        if_exists="append",  # adds rows, doesn't overwrite
        index=False
    )
    print(f"Saved {len(df)} row(s) to {db_path}")

load(clean_df)

# Query it back to verify
import sqlite3
conn = sqlite3.connect("weather.db")
result = pd.read_sql("SELECT * FROM weather_readings", conn)
print(result)

Saved 1 row(s) to weather.db
    city  temp_f  feels_like_f  humidity         condition  wind_mph  \
0  Tampa   84.76          87.4        55  scattered clouds      13.8   
1  Tampa   84.76          87.4        55  scattered clouds      13.8   

                   fetched_at  
0  2026-04-27T22:41:46.225428  
1  2026-04-27T22:45:28.918891  


In [5]:
# pipeline.py — full ETL in one function
def run_pipeline(city: str):
    print(f"Running ETL for {city}...")
    raw = extract(city)
    clean = transform(raw)
    load(clean)
    print("Done!")

# Schedule with the 'schedule' library
import schedule, time

schedule.every(1).hour.do(run_pipeline, city="Tampa,FL,US")

if __name__ == "__main__":
    run_pipeline("Tampa,FL,US")  # run once on start
    while True:
        schedule.run_pending()
        time.sleep(60)

# Or use cron on Mac/Linux (no extra library)
# crontab -e → add:
# 0 * * * * /usr/bin/python3 /path/to/pipeline.py

Running ETL for Tampa,FL,US...
Saved 1 row(s) to weather.db
Done!


KeyboardInterrupt: 